# Duck ProgramIR LLM Code-Generation Smoke Test

This Kaggle notebook verifies the complete production path: a real Qwen model must generate a schema-constrained `python` tool call, Duck must validate and lower its ProgramIR, and the hardened sandbox must execute the generated program with the expected result.

Success requires the model to build a generator comprehension that computes the sum of the squares of the even integers from 1 through 10. A hard-coded result or malformed IR fails the notebook. Internet access is not required; the source, model, and vLLM wheelhouse are attached as Kaggle datasets.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
WORKING_DIR = Path('/kaggle/working/duck-programir-codegen-smoke')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

def dataset_path(slug: str) -> Path:
    direct = INPUT_ROOT / slug
    if direct.exists():
        return direct
    matches = sorted(path for path in INPUT_ROOT.glob(f'*{slug}*') if path.is_dir())
    if len(matches) == 1:
        return matches[0]
    raise FileNotFoundError(f'Expected one attached Kaggle dataset matching {slug!r}; found {matches}')

SOURCE_DATASET = dataset_path('taaf-kaggle-source')
WHEELHOUSE = dataset_path('arc3-vllm-h100-wheelhouse-v3')
MODEL_DATASET = dataset_path('vrfai-qwen3-6-27b-fp8-hf-snapshot')
ARC3_SOURCE = SOURCE_DATASET / 'src' / 'ARC3-Inference'
if not ARC3_SOURCE.exists():
    raise FileNotFoundError(f'Duck source bundle is missing: {ARC3_SOURCE}')
sys.path.insert(0, str(ARC3_SOURCE))

SETUP_ENV_PATH = WORKING_DIR / 'setup-env.json'
SETUP_ENV_PATH.write_text('{}', encoding='utf-8')
os.environ.update({
    'TAAF_KAGGLE_WORKING_DIR': str(WORKING_DIR),
    'TAAF_KAGGLE_SETUP_ENV': str(SETUP_ENV_PATH),
    'TAAF_KAGGLE_INPUT_PATHS': json.dumps({
        'driessmit1/arc3-vllm-h100-wheelhouse-v3': str(WHEELHOUSE),
        'driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot': str(MODEL_DATASET),
    }),
})
print('Duck source:', ARC3_SOURCE)
print('vLLM wheelhouse:', WHEELHOUSE)
print('Qwen model:', MODEL_DATASET)

In [ ]:
from inference.framework.kaggle import (
    DuckKaggleVllmConfig,
    duck_kaggle_setup_command,
)

config = DuckKaggleVllmConfig()
setup_env = os.environ.copy()
setup_env['PYTHON'] = sys.executable
subprocess.run(
    ['bash', '-lc', duck_kaggle_setup_command(config)],
    check=True,
    env=setup_env,
)
runtime_env = json.loads(SETUP_ENV_PATH.read_text(encoding='utf-8'))
os.environ.update({str(key): str(value) for key, value in runtime_env.items()})
BASE_URL = runtime_env['LOCAL_ANALYZER_BASE_URL'].rstrip('/')
MODEL_ID = runtime_env['LOCAL_ANALYZER_MODEL_ID']
print('Ready:', BASE_URL, MODEL_ID)

In [ ]:
import requests

from inference.agent.program_ir import program_tool_parameters_schema

tools = [{
    'type': 'function',
    'function': {
        'name': 'python',
        'description': 'Compile and execute structured Duck ProgramIR.',
        'parameters': program_tool_parameters_schema(),
        'strict': True,
    },
}]
prompt = '''Emit exactly one python tool call. Build ProgramIR version 1 that assigns result to the sum of the squares of every even integer from 1 through 10 inclusive. You must compute it using a generator_comprehension passed to sum, with range, multiplication, modulo, and an equality comparison. Do not hard-code 220. Do not explain the answer.'''
request_payload = {
    'model': MODEL_ID,
    'messages': [
        {'role': 'system', 'content': 'You generate valid Duck ProgramIR tool arguments. Follow the supplied schema exactly.'},
        {'role': 'user', 'content': prompt},
    ],
    'tools': tools,
    'tool_choice': 'required',
    'temperature': 0.0,
    'top_p': 1.0,
    'top_k': 1,
    'max_tokens': 4096,
    'chat_template_kwargs': {'enable_thinking': False},
}
response = requests.post(
    f'{BASE_URL}/chat/completions',
    json=request_payload,
    timeout=300,
)
response.raise_for_status()
response_payload = response.json()
message = response_payload['choices'][0]['message']
tool_calls = message.get('tool_calls') or []
assert len(tool_calls) == 1, f'Expected exactly one tool call, got: {message}'
function = tool_calls[0].get('function') or {}
assert function.get('name') == 'python', function
raw_arguments = function.get('arguments')
arguments = json.loads(raw_arguments) if isinstance(raw_arguments, str) else raw_arguments
assert isinstance(arguments, dict) and isinstance(arguments.get('program'), dict), arguments
generated_program = arguments['program']
generated_json = json.dumps(generated_program, indent=2, ensure_ascii=False)
result_assignments = [
    statement for statement in generated_program.get('body', [])
    if statement.get('kind') == 'assign'
    and any(target.get('kind') == 'name_target' and target.get('name') == 'result' for target in statement.get('targets', []))
]
assert len(result_assignments) == 1, 'Model must assign result exactly once.'
result_value = result_assignments[0].get('value') or {}
assert result_value.get('kind') == 'call', 'result must be produced by a call to sum.'
assert (result_value.get('function') or {}).get('kind') == 'name'
assert result_value['function'].get('name') == 'sum', 'result must call sum.'
generator_args = [arg for arg in result_value.get('args', []) if arg.get('kind') == 'generator_comprehension']
assert len(generator_args) == 1, 'sum must receive exactly one generator_comprehension.'
canonical_generated = json.dumps(generated_program, sort_keys=True, separators=(',', ':'))
for required_fragment in ('"op":"mul"', '"op":"mod"', '"ops":["eq"]'):
    assert required_fragment in canonical_generated, f'Missing required computation node: {required_fragment}'
print('MODEL-GENERATED PROGRAMIR')
print(generated_json)

In [ ]:
from inference.agent.program_ir import COMPILER_VERSION, compile_program
from inference.agent.python_tool_sandbox import run_sandboxed_python

compiled = compile_program(generated_program)
execution = run_sandboxed_python(
    code=compiled.source,
    timeout_seconds=10,
    initial_state={},
    action_handler=lambda actions: {},
    strategy_handler=lambda update: update,
)
if execution.get('error'):
    raise AssertionError(f'Generated code failed in the Duck sandbox: {execution}')
assert execution.get('result') == 220, f'Expected 220, got: {execution}'

report = {
    'status': 'PASS',
    'model': MODEL_ID,
    'compiler_version': COMPILER_VERSION,
    'compiler': compiled.metadata(),
    'result': execution['result'],
    'lowered_source': compiled.source,
    'generated_program': generated_program,
}
report_path = Path('/kaggle/working/duck_programir_codegen_smoke.json')
report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
print('LOWERED PYTHON')
print(compiled.source)
print(f'PASS: Qwen generated executable ProgramIR; result={execution["result"]}')
print('Report:', report_path)

## Reading the result

The notebook passes only if the model emitted a valid `python` tool call, used the requested generator-comprehension structure, passed Duck's schema/safety/compiler pipeline, executed in the sandbox, and returned `220`. The final JSON report is written to `/kaggle/working/duck_programir_codegen_smoke.json`.